In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch_geometric

import sys
sys.path.append('./..')
    
from ld_gcn import network, loader, plotting, preprocessing, testing, error, training, utils, initialization
from IPython.display import HTML

import numpy as np

# Define PDE problem

In [ ]:
pde_problem = 11
problem_name, variable, mu_space, n_param, dim_pde, n_comp, n_sim, HyperParams = utils.prepare_HyperParams(pde_problem)

# Initialize device and set reproducibility

In [ ]:
device = initialization.initialize(HyperParams)

# Load dataset

In [ ]:
dataset_dir = '../dataset/'+problem_name+'_unstructured.mat'
dataset = loader.LoadDataset(dataset_dir, variable, dim_pde, n_comp)

In [ ]:
n_snap2keep = int(dataset.U.shape[1]/(len(mu_space[0])*len(mu_space[1])))
delete_first_n = 1

dataset, mu_space = preprocessing.delete_initial_condition(dataset, mu_space, n_comp, n_snap2keep, n_delete=delete_first_n)

params = utils.create_param_list(mu_space, device)

In [ ]:
graph_loader, train_loader, test_loader, \
    val_loader, scaler_all, scaler_test, xyz, VAR_all, VAR_val, VAR_test, \
        train_trajectories, val_trajectories, test_trajectories, params_train, params_test = preprocessing.graphs_dataset(dataset, HyperParams, params)

# Define the architecture

In [ ]:
RecNet = network.RecNet(HyperParams)
RecNet = RecNet.to(device)
DynNet = network.DynNet(HyperParams)
DynNet = DynNet.to(device)

torch.set_default_dtype(torch.float32)

optimizer = 'ADAM' # 'ADAM' or 'LBFGS'

if optimizer == 'ADAM':
    optimizer = torch.optim.Adam([
        {'params': DynNet.parameters()},
        {'params': RecNet.parameters()}
      ],
      lr=HyperParams.learning_rate,
      weight_decay=HyperParams.weight_decay
    )

elif optimizer == 'LBFGS':
    optimizer = torch.optim.LBFGS(
        list(DynNet.parameters())+list(RecNet.parameters()),
        lr = 1.,
        max_iter = HyperParams.max_epochs,
        max_eval = None,
        tolerance_grad = 1e-07,
        tolerance_change = 1e-09,
        history_size = 30,
        line_search_fn = 'strong_wolfe',
    )
    
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=HyperParams.miles, gamma=HyperParams.gamma)

In [ ]:
compile = False

if compile:
    try:
        import torch._dynamo
        torch._dynamo.config.suppress_errors = True
        RecNet = torch.compile(RecNet)
        DynNet = torch.compile(DynNet)
        print('The networks have been compiled successfully')
    except:
        print('Not possible to compile the networks')

In [ ]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters in dynnet:", count_trainable_params(DynNet))
print("Trainable parameters in recnet:", count_trainable_params(RecNet))

# Train or load a pre-trained network

In [ ]:
# To reduce memory consumption on GPU:
params = params.to("cpu")
VAR_all = VAR_all.to("cpu")
VAR_val = VAR_val.to("cpu")
VAR_test = VAR_test.to("cpu")
if device=='cuda':
    torch.cuda.empty_cache()

In [ ]:
load = True
train = False

if load:
    try:
        RecNet.load_state_dict(torch.load(HyperParams.net_dir+HyperParams.net_name+HyperParams.net_run+'_decoder.pt', map_location=torch.device('cpu')))
        DynNet.load_state_dict(torch.load(HyperParams.net_dir+HyperParams.net_name+HyperParams.net_run+'_dyn.pt', map_location=torch.device('cpu')))
        print('Loading saved network')
    except FileNotFoundError:
        print('Not possible to load the network')
        train = True
else:
    train = True

if train:
    training.train(RecNet, DynNet, optimizer, device, scheduler, train_loader, test_loader, HyperParams, params_train, params_test)

# Evaluate the model

In [ ]:
RecNet = RecNet.to("cpu")
DynNet = DynNet.to("cpu")

vars = "GCA-ROM"
VAR_train = VAR_all[train_trajectories,:,:]

In [ ]:
results, latents = testing.evaluate(VAR_all, RecNet, DynNet, graph_loader, params, HyperParams)
results_test, latents_test = testing.evaluate(VAR_test, RecNet, DynNet, test_loader, params_test, HyperParams)
results_train, latents_ = testing.evaluate(VAR_train, RecNet, DynNet, train_loader, params_train, HyperParams)
results_val = results[val_trajectories,:,:]

# Compute the errors

In [ ]:
vars = problem_name

error_abs, norm = error.compute_error(results, VAR_all, scaler_all)
error_abs_test, norm_test = error.compute_error(results_test, VAR_test, scaler_all)
error_abs_train, norm_train = error.compute_error(results_train, VAR_train, scaler_all)
error_abs_val, norm_val = error.compute_error(results_val, VAR_val, scaler_all)

print('\nERRORS ON THE TRAINING SET:')
error.print_error(error_abs_train, norm_train, vars)
print('\nERRORS ON THE TEST SET:')
error.print_error(error_abs_test, norm_test, vars)
print('\nERRORS ON THE VALIDATION DATASET:')
error.print_error(error_abs_val, norm_val, vars)
print('\nERRORS ON THE WHOLE DATASET:')
error.print_error(error_abs, norm, vars)

error.save_error(error_abs_test, norm_test, HyperParams, vars)

In [ ]:
n_snapshots = n_snap2keep - delete_first_n

max_error_index = np.argmax(np.array(error_abs)/np.array(norm))
print(f'Index of the max of the relative error: {max_error_index}')
print(f'Indices of test trajectories: {(np.array(test_trajectories)/(n_snapshots))[::n_snapshots]}')

# Plot the results

In [ ]:
plotting.plot_loss(HyperParams)

SAMPLE = 3
plotting.plot_latent_time(HyperParams, SAMPLE, latents, params, n_sim)

component = 2
plotting.plot_latent_component(HyperParams, component, latents, params, n_sim)

total_times = n_snap2keep - delete_first_n
SAMPLE = max_error_index//total_times # index of the parameter for which we want to plot, in the case the one with the greatest error
SNAP = max_error_index - SAMPLE*total_times #SNAP represents the i-th snapshot in the time evolution

plotting.plot_fields(SAMPLE, SNAP, results, scaler_all, HyperParams, dataset, params)

In [ ]:
final_time = params_train[..., -1][0][-1].to(device='cpu')
rel_errors = np.array(error_abs)/np.array(norm)
rel_errors_test = np.array(error_abs_test)/np.array(norm_test)

#plotting.plot_relative_errors_vs_time(rel_errors, params, HyperParams, final_time, flag='all')
#plotting.plot_relative_errors_vs_time(rel_errors_test, params_test, HyperParams, final_time, flag='test')

In [ ]:
component = 0 # 8 is nice if n=15
plotting.plot_latent_by_mu_group(HyperParams, component, latents, params, n_sim, mu_index=0)
plotting.plot_latent_by_mu_group(HyperParams, component, latents, params, n_sim, mu_index=1)

plot_all = True

if plot_all:
    for i in range(5):
        for j in range(2):
            plotting.plot_latent_by_mu_group(HyperParams, component, latents, params, n_sim, mu_index=j, single_mu_index=i)

In [ ]:
n_times_fine = int(latents.shape[0]/n_sim)
params_np = params.detach().cpu().numpy()
latents_np = latents.detach().cpu().numpy()

#plotting.animate_latent_evolution(latents_np, params_np, component, n_sim, n_times_fine, HyperParams, three_d=False)

In [ ]:
has_errors = True
if has_errors:
    errors_gca = np.load('../gca_data/errors_gca_MH.npy')
    norms_gca = norm_test
    rel_errors_gca = errors_gca/norms_gca
    plotting.compare_errors_gca(rel_errors_test, rel_errors_gca, params_test, HyperParams, final_time)

In [ ]:
has_fields = True if HyperParams.bottleneck_dim == 15 else False
#if not isinstance(Z, np.ndarray):
#    Z = Z.detach().numpy()

if has_fields:
    gca_fields = np.load(f'../gca_data/scaled_output_MH_15.npy') # to be changed
    Z = preprocessing.inverse_normalize_input(results.detach()[:,:,0].T, scaler_all)
    Z = Z.numpy() # same shape as gca_fields: (n_nodes, n_snaps)

    FO_fields = preprocessing.inverse_normalize_input(VAR_all[:,:,0].T, scaler_all)
    FO_fields = FO_fields.numpy() # same shape as gca_fields

    SAMPLE = 10
    print('Plotting the first field...')
    plotting.create_animation(SAMPLE, Z, HyperParams, dataset, xyz, n_sim, comp="_U", flag='sim')
    print('Plotting the second field...')
    plotting.create_animation(SAMPLE, gca_fields, HyperParams, dataset, xyz, n_sim, comp="_U", flag='gca')
    print('Plotting the third field...')
    plotting.create_animation(SAMPLE, FO_fields, HyperParams, dataset, xyz, n_sim, comp="_U", flag='h')

In [ ]:
import torch
import gc

def get_gpu_tensors(min_size_mb=1.0):
    tensors = []
    total_mem = 0

    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                size_bytes = obj.element_size() * obj.nelement()
                size_mb = size_bytes / 1024**2
                if size_mb >= min_size_mb:
                    tensors.append((obj, size_mb))
                    total_mem += size_bytes
        except Exception:
            pass

    print(f"Total GPU memory (tensors ≥ {min_size_mb:.1f} MB): {total_mem / 1024**2:.2f} MB")
    print(f"Number of tensors ≥ {min_size_mb:.1f} MB: {len(tensors)}")

    for i, (t, size_mb) in enumerate(tensors):
        shape = tuple(t.shape)
        dtype = t.dtype
        print(f"{i:03d}: {shape} | {dtype} | {size_mb:.2f} MB")
    
    return [t[0] for t in tensors]

#get_gpu_tensors()